# Reduced MultiTask GP

高次元入力をPCAで圧縮しながら task identity を保持する MultiTask / Kronecker の考え方を確認します。


## Long-format MultiTask


In [ ]:
import torch
from robotorchan.models import PCAMultiTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X0 = torch.rand(8, 5)
X1 = torch.rand(8, 5)
train_X = torch.cat([
    torch.cat([X0, torch.zeros(8, 1)], dim=-1),
    torch.cat([X1, torch.ones(8, 1)], dim=-1),
])
train_Y = torch.cat([X0[:, :1] + X0[:, 1:2], X1[:, :1] - X1[:, 1:2]])
model = PCAMultiTaskGP(train_X, train_Y, task_feature=-1, n_components=2)
print(model.raw_train_X.shape)


## Posterior


In [ ]:
X_test = torch.cat([torch.rand(3, 5), torch.zeros(3, 1)], dim=-1)
print(model.posterior(X_test).mean.squeeze(-1))


## Fit the multi-task GP

タスク列を保持したまま、既存のExact MLLで学習します。


In [ ]:
from botorch.fit import fit_gpytorch_mll

fit_gpytorch_mll(model.make_mll())
model.eval()
print("fitted task-aware model")


## Task-specific candidate proposals

各タスクを固定した候補プールを元の5次元空間で評価し、posterior mean最大の候補を提案します。タスクIDは最適化対象ではありません。


In [ ]:
candidate_features = torch.rand(128, 5)
for task_id in (0, 1):
    task_column = torch.full((128, 1), float(task_id))
    pool = torch.cat([candidate_features, task_column], dim=-1)
    with torch.no_grad():
        scores = model.posterior(pool).mean.squeeze(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    features = candidate[:, :5]
    if task_id == 0:
        observed = features[:, :1] + features[:, 1:2]
    else:
        observed = features[:, :1] - features[:, 1:2]
    assert candidate.shape == torch.Size([1, 6])
    assert candidate[0, -1].item() == task_id
    print("task:", task_id, "candidate:", candidate)
    print("posterior mean:", scores.max(), "objective:", observed)
